# Vietnamese Budget Forcing — Kaggle Runner

**Research question:** Does Test-Time Scaling via Budget Forcing transfer to Vietnamese language reasoning?

**Before running:**
1. Settings → Accelerator → **GPU T4 x1** (or P100)
2. Settings → Internet → **On**
3. Add your HuggingFace token: Settings → Secrets → Add `HF_TOKEN`

**Run order:** Cell 1 (GPU check) → Cell 2 (clone) → Cell 3 (install) → Cell 4 (HF auth) → Cell 5 (validate benchmarks) → Cell 6 (VRAM check) → Cell 7 (run sweep) → Cell 8 (aggregate) → Cell 9 (copy outputs)

**Scope:** BF-only, n_wait ∈ {0,1,2}. No retrieval.

In [ ]:
# Cell 1 — Check GPU and environment
import subprocess, os, sys

print('=== GPU ===')
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv'], check=False)

import torch
print(f'\nPyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

WORK_DIR = '/kaggle/working'
print(f'\nWorking dir: {WORK_DIR}')
print('Disk free: ', end='')
subprocess.run(['df', '-h', WORK_DIR])

In [ ]:
# Cell 2 — Clone repo
import os

REPO_DIR = '/kaggle/working/s1_budget_forcing'

if not os.path.exists(REPO_DIR):
    # Option A: clone from GitHub (if repo is public)
    # !git clone https://github.com/YOUR_USERNAME/s1_budget_forcing.git {REPO_DIR}

    # Option B: upload repo as a Kaggle Dataset and copy
    # The dataset would be at /kaggle/input/s1-budget-forcing/
    import shutil
    INPUT_PATH = '/kaggle/input/s1-budget-forcing'
    if os.path.exists(INPUT_PATH):
        shutil.copytree(INPUT_PATH, REPO_DIR)
        print(f'Copied from Kaggle dataset to {REPO_DIR}')
    else:
        raise FileNotFoundError(
            'Repo not found. Either:\n'
            '  A) Uncomment the git clone line above, or\n'
            '  B) Upload the repo as a Kaggle dataset named "s1-budget-forcing"'
        )
else:
    print(f'Repo already at {REPO_DIR}')

os.chdir(REPO_DIR)
print(f'CWD: {os.getcwd()}')
!ls

In [ ]:
# Cell 3 — Install dependencies
# Kaggle has torch/transformers pre-installed; only install what's missing.
# No RAG dependencies needed — BF-only study.
!pip install -q \
    datasets>=2.18.0 \
    bitsandbytes>=0.43.0 \
    accelerate>=0.27.0 \
    tqdm

print('\nInstallation complete.')

In [ ]:
# Cell 4 — HuggingFace authentication
# Required for r1-distill-7B and Vietnamese models.
# Store your token in Kaggle Secrets as HF_TOKEN — never hardcode it here.
from kaggle_secrets import UserSecretsClient
import os

try:
    secret = UserSecretsClient()
    hf_token = secret.get_secret('HF_TOKEN')
    os.environ['HF_TOKEN'] = hf_token
    from huggingface_hub import login
    login(token=hf_token, add_to_git_credential=False)
    print('HuggingFace login successful.')
except Exception as e:
    print(f'[WARN] HF auth skipped: {e}')
    print('Public models (Qwen2.5, SeaLLM) will still work without a token.')

In [ ]:
# Cell 5 — Validate Vietnamese benchmarks
import sys, os
sys.path.insert(0, '/kaggle/working/s1_budget_forcing/experiments')
os.chdir('/kaggle/working/s1_budget_forcing')

!python experiments/data/download_vi_benchmarks.py --n_samples 2

In [ ]:
# Cell 6 — VRAM check
# Verify each model fits in T4 VRAM (15 GB) under 4-bit quantization.
# r1-distill-7B and Vi-7B models: ~3.5 GB each at 4-bit.
# Run this before the sweep to catch OOM before wasting GPU time.
import sys, os
sys.path.insert(0, '/kaggle/working/s1_budget_forcing/experiments')
os.chdir('/kaggle/working/s1_budget_forcing')

from models.model_loader import SUPPORTED_MODELS, estimate_vram_gb
import torch

VI_MODELS = ['qwen2.5-3B', 'r1-distill-7B', 'vinallama-7b', 'vistral-7b', 'seallm-7b']

if torch.cuda.is_available():
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU VRAM: {total_vram:.1f} GB')
else:
    total_vram = 0
    print('[WARN] No GPU detected — models will run on CPU (very slow)')

print()
print(f'{"Model":<20} {"HF ID":<45} {"VRAM 4-bit":>10} {"Fits T4":>8}')
print('-' * 90)
for name in VI_MODELS:
    hf_id = SUPPORTED_MODELS.get(name, 'NOT IN REGISTRY')
    vram = estimate_vram_gb(name, load_in_4bit=True)
    fits = '✓' if vram < total_vram * 0.9 else '✗ OOM risk'
    print(f'{name:<20} {hf_id:<45} {vram:>8.1f} GB {fits:>8}')

print()
print('Note: Vinallama/Vistral use </s> as EoT token (LLaMA-2/Mistral base).')
print('      If thinking_tokens stays 0 after n_wait>0, check EoT suppression in decoding.py.')

In [ ]:
# Cell 7 — Run experiments
#
# Set PRESET below, then Run All from this cell.
# Start with SMOKE to confirm pipeline, then scale up.
#
# SMOKE        1 model × 1 benchmark × 3 n_wait × 5 samples  → ~5 min
# SMALL_MATRIX 2 models × 2 benchmarks × 3 n_wait × 20 samples → ~30-45 min
# FULL         5 models × 2 benchmarks × 3 n_wait × 100 samples → ~3-5 hr

import os, subprocess
os.chdir('/kaggle/working/s1_budget_forcing')

# ── Choose a preset ──────────────────────────────────────────────────────────
PRESET = 'SMOKE'   # 'SMOKE' | 'SMALL_MATRIX' | 'FULL'

PRESETS = {
    'SMOKE': dict(
        MODELS      = 'qwen2.5-3B',
        BENCHMARKS  = 'vi_gsm8k',
        N_WAIT_LIST = '0 1 2',
        N_SAMPLES   = '5',
        EXTRA_ARGS  = '--max_tokens 512',   # 4-bit ON by default on Kaggle GPU
    ),
    'SMALL_MATRIX': dict(
        MODELS      = 'qwen2.5-3B r1-distill-7B',
        BENCHMARKS  = 'vi_gsm8k vimmlu',
        N_WAIT_LIST = '0 1 2',
        N_SAMPLES   = '20',
        EXTRA_ARGS  = '',
    ),
    'FULL': dict(
        MODELS      = 'qwen2.5-3B r1-distill-7B vinallama-7b vistral-7b seallm-7b',
        BENCHMARKS  = 'vi_gsm8k vimmlu',
        N_WAIT_LIST = '0 1 2',
        N_SAMPLES   = '100',
        EXTRA_ARGS  = '',
    ),
}

cfg = PRESETS[PRESET]
print(f'Running preset: {PRESET}')
for k, v in cfg.items():
    print(f'  {k}={v!r}')

env = os.environ.copy()
env.update({
    'MODELS':      cfg['MODELS'],
    'BENCHMARKS':  cfg['BENCHMARKS'],
    'N_WAIT_LIST': cfg['N_WAIT_LIST'],
    'N_SAMPLES':   cfg['N_SAMPLES'],
    'OUTPUT_DIR':  'experiments/results',
    'EXTRA_ARGS':  cfg['EXTRA_ARGS'],
})

result = subprocess.run(
    ['bash', 'experiments/scripts/run_vi_bf.sh'],
    env=env,
    cwd='/kaggle/working/s1_budget_forcing',
)
print(f'\nExit code: {result.returncode}')

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/s1_budget_forcing'

In [3]:
# Cell 8 — Aggregate results
import glob, os

results_root = '/kaggle/working/s1_budget_forcing/experiments/results'
run_dirs = sorted(glob.glob(f'{results_root}/vi_*/'))

if not run_dirs:
    print('No vi_* result directories found. Run Cell 7 first.')
else:
    latest = run_dirs[-1]
    print(f'Aggregating: {latest}')
    !python experiments/results/summary_vi.py --results_dir {latest}

    import pandas as pd
    csv_path = os.path.join(latest, 'summary_vi.csv')
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        display_cols = ['model', 'benchmark', 'n_wait', 'n_samples',
                        'accuracy', 'scaling', 'performance',
                        'avg_thinking_tokens', 'extraction_failures']
        print(df[display_cols].to_string(index=False))
    else:
        print(f'CSV not found at {csv_path}')

No vi_* result directories found. Run Cell 7 first.


In [ ]:
# Cell 9 — Copy outputs to /kaggle/working for download
# Kaggle allows downloading files from /kaggle/working/.
import shutil, os, glob

OUT = '/kaggle/working/outputs'
os.makedirs(OUT, exist_ok=True)

for d in glob.glob('/kaggle/working/s1_budget_forcing/experiments/results/vi_*'):
    dest = os.path.join(OUT, os.path.basename(d))
    if not os.path.exists(dest):
        shutil.copytree(d, dest)
        print(f'Copied: {dest}')

print(f'\nAll outputs in: {OUT}')
!ls -lh {OUT}